In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import numpy as np
import shapely

from gridr.core.grid import grid_commons

from notebook_utils import plot_convention_grid_mesh, mpl_plot_wrapper

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# Shared raster geometry used throughout the masking tutorial series.
shape, resolution = (6, 8), (1, 1)
origin_int, origin_half = (0., 0.), (0.5, 0.5)

# Grid-coordinate arrays for both conventions, used by plotting helpers.
cxx_int,  cyy_int  = grid_commons.grid_regular_coords_2d(shape, origin_int,  resolution, sparse=False)
cxx_half, cyy_half = grid_commons.grid_regular_coords_2d(shape, origin_half, resolution, sparse=False)

# The `grid_rasterize` module

This notebook introduces `grid_rasterize`, GridR's low-level rasterisation utility. It converts a Shapely geometry into a binary raster aligned with a target coordinate frame, and powers the higher-level `build_mask` API covered in the next notebooks.

**What you'll learn**

- The two ways to define a target frame: `grid_coords` vs.
  `(shape, origin, resolution)`
- The `inner_value` / `outer_value` / `default_value` semantics
- How to limit the computation to a sub-window
- The available rasterisation algorithms in GridR

## Setting things up

GridR currently integrates two backends for rasterisation: `shapely` and `rasterio`. The `shapely` path is still available in the codebase but is significantly outperformed by `rasterio`, so we use the `rasterio` algorithm exclusively here. Note that no Rust rasterisation backend exists yet.

## Calling `grid_rasterize`

We rasterise a polygon defined in the half-real convention against the shared `(6, 8)` raster. Two target-frame options exist:

- pass a pre-built coordinate array through `grid_coords`, or
- pass the triplet `(shape, origin, resolution)`.

Here we use the second option. The `origin` argument expects the *geometry*'s origin (the rasteriser's internal convention matches GridR's).

In [ ]:
from gridr.core.grid import grid_rasterize

# Rasterio-based rasterisation algorithm.
alg = grid_rasterize.GridRasterizeAlg.RASTERIO_RASTERIZE
kwargs_alg = {}

# Burn-in values.
inner_value, outer_value, default_value = 1, 2, 0

geometry_origin = (0.5, 0.5)
geometry = shapely.geometry.Polygon([
    (3.5, 2.5),
    (6.5, 2.5),
    (6.5, 4.5),
    (3.5, 4.5),
])

raster = grid_rasterize.grid_rasterize(
    grid_coords   = None,
    shape         = shape,
    origin        = geometry_origin,
    resolution    = resolution,
    win           = None,
    inner_value   = inner_value,
    outer_value   = outer_value,
    default_value = default_value,
    geometry      = geometry,
    alg           = alg,
    dtype         = np.uint8,
    **kwargs_alg,
)

In [ ]:
value_color_alpha_map = (
    (inner_value,   "orange", 0.2),
    (outer_value,   "blue",   0.2),
    (default_value, "grey",   0.2),
)
plot_convention_grid_mesh(
    shape, resolution, origin_int, cxx_int, cyy_int,
    geometry=geometry, geometry_origin=geometry_origin,
    mask=raster, win=None,
    value_color_alpha_map=value_color_alpha_map,
    prefix="grid_rasterize_geometry",
    title="Rasterize geometry",
)

Pixels whose centroids fall inside or on the contour of the geometry are *burned* with `inner_value` (orange). All other pixels receive `outer_value` (blue). `default_value` is only used when no geometry at all is provided (passing `geometry=[]`).

### Empty geometry

Passing an empty list `[]` triggers the `default_value` path -- every pixel is burned with `default_value`. Passing `None` instead would raise an exception.

In [ ]:
raster = grid_rasterize.grid_rasterize(
    grid_coords   = None,
    shape         = shape,
    origin        = geometry_origin,
    resolution    = resolution,
    win           = None,
    inner_value   = inner_value,
    outer_value   = outer_value,
    default_value = default_value,
    geometry      = [],
    alg           = alg,
    dtype         = np.uint8,
    **kwargs_alg,
)

In [ ]:
plot_convention_grid_mesh(
    shape, resolution, origin_int, cxx_int, cyy_int,
    geometry=geometry, geometry_origin=geometry_origin,
    mask=raster, win=None,
    value_color_alpha_map=value_color_alpha_map,
    prefix="grid_rasterize_no_geometry",
    title="Rasterize with no geometry defined",
)

### Limiting computation with `win`

`win` restricts the rasterisation to a sub-region of the full target grid. It follows GridR's window convention -- a list of `(start, end)` *inclusive* index pairs, row first then column.

The output array shape and values reflect only the requested window.

In [ ]:
win = np.array([(1, 3), (5, 7)])
raster = grid_rasterize.grid_rasterize(
    grid_coords   = None,
    shape         = shape,
    origin        = geometry_origin,
    resolution    = resolution,
    win           = win,
    inner_value   = inner_value,
    outer_value   = outer_value,
    default_value = default_value,
    geometry      = geometry,
    alg           = alg,
    dtype         = np.uint8,
    **kwargs_alg,
)
display(raster)

The next three notebooks build on top of `grid_rasterize` to demonstrate `build_mask`, the higher-level API for combining geometry-based and raster-based masks into a single binary mask.